# 직캠 스테이지 · 코랩으로 누끼 따기

직캠에서 설윤만 따서, 앱에 바로 넣을 수 있는 클립 파일(`이름.stage-tb.mp4`)을 만들어요.
코랩의 GPU로 돌려서 노트북 PC보다 훨씬 빠르고, 더 큰 모델(SAM 2.1 base+, RVM ResNet-50)을 써서 더 정확해요.

**쓰는 법**
1. 위 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**를 골라요.
2. 아래 칸을 위에서부터 차례로 ▶ 눌러 실행해요. (**런타임 → 모두 실행**도 돼요.)
3. 다 되면 내 구글 드라이브의 **직캠 스테이지/코랩** 폴더에 파일이 생겨요.
   PC나 폰에서 받아서 앱의 **파일 가져오기**로 넣으면 끝이에요.

설윤 사진을 드라이브 폴더에 3~5장 넣어 두면 그걸로 찾고, 없으면 영상에서 가장 많이 나온 얼굴(= 직캠 주인공)로 찾아요.

## 1. GPU 확인과 설치 (처음 한 번 2~3분)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "GPU가 없어요: 런타임 → 런타임 유형 변경 → T4 GPU"
import os
os.environ['SAM2_BUILD_CUDA'] = '0'  # skip the optional CUDA extension (slow to build, not needed)
!pip -q install sam2 yt-dlp onnxruntime-gpu

## 2. 구글 드라이브 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. 설정

오른쪽 칸을 채우고 ▶를 누르세요.

In [ ]:
VIDEO = 'https://youtu.be/tE33xh6wdEI'  #@param {type:"string"}
#@markdown ↑ 유튜브 링크, 또는 드라이브 안의 영상 경로 (예: `/content/drive/MyDrive/직캠/설윤.mp4`)
NAME = '설윤'  #@param {type:"string"}
PHOTOS_FOLDER = '/content/drive/MyDrive/직캠 스테이지/설윤 사진'  #@param {type:"string"}
#@markdown ↑ 혼자 나온 정면 사진 3~5장이 든 폴더 (없으면 비워 두세요)
START = 0  #@param {type:"number"}
END = 0  #@param {type:"number"}
#@markdown ↑ 초 단위, END가 0이면 끝까지
SHAPE = '누끼'  #@param ["누끼", "원"]
QUALITY = '고화질 (긴 변 1280)'  #@param ["보통 (긴 변 960)", "고화질 (긴 변 1280)", "FHD (긴 변 1920)"]
FPS = 30  #@param [24, 30, 60] {type:"raw"}
SAM_MODEL = 'base_plus'  #@param ["tiny", "small", "base_plus", "large"]
OUT_FOLDER = '/content/drive/MyDrive/직캠 스테이지/코랩'

## 4. 처리 코드

In [ ]:
%%writefile /content/fancam.py
"""직캠 스테이지 for Google Colab: cuts one member out of a fancam on a cloud GPU.

The same idea as the web app (js/processor.js), with the full-size models it can't run in a
browser: SAM 2.1 (base+, all 7 memory frames) follows her, Robust Video Matting (ResNet-50) draws
her edges, and face recognition (InsightFace buffalo_s) keeps the tracker on her. The result is
the app's own clip format (colour | alpha packed in one MP4, "<name>.stage-<layout>.mp4"), so the
app's "파일 가져오기" takes it as is.

Long videos don't fit in memory at once, so SAM 2 runs in chunks, each starting from the last
mask of the one before; when the face check finds that the tracker has drifted onto someone else,
the chunk ends there and the next one starts again from her face.
"""
import json
import math
import os
import shutil
import subprocess
import urllib.request
import zipfile

import cv2
import numpy as np
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# bfloat16 halves SAM 2's time on GPUs that have it (Ampere and newer; Colab's T4 doesn't).
BF16 = DEVICE == 'cuda' and torch.cuda.is_bf16_supported(including_emulation=False)
CACHE = os.path.expanduser('~/.cache/fancam')

SAM_MODELS = {  # name -> (config, checkpoint URL)
    'tiny': ('configs/sam2.1/sam2.1_hiera_t.yaml', 'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt'),
    'small': ('configs/sam2.1/sam2.1_hiera_s.yaml', 'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt'),
    'base_plus': ('configs/sam2.1/sam2.1_hiera_b+.yaml', 'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_base_plus.pt'),
    'large': ('configs/sam2.1/sam2.1_hiera_l.yaml', 'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt'),
}
FACE_ZIP = 'https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_s.zip'


def fetch(url, name=None):
    os.makedirs(CACHE, exist_ok=True)
    path = os.path.join(CACHE, name or url.rsplit('/', 1)[-1])
    if not os.path.exists(path):
        print('받는 중:', url)
        tmp = path + '.part'
        urllib.request.urlretrieve(url, tmp)
        os.replace(tmp, path)
    return path


# ---------------------------------------------------------------------------
# Faces (SCRFD-500M + MobileFaceNet, the same models and maths as js/face.js)

TEMPLATE = np.array([[38.2946, 51.6963], [73.5318, 51.5014], [56.0252, 71.7366], [41.5493, 92.3655], [70.7299, 92.2041]], np.float32)
SAME, MARGIN, OTHER, OTHER_LEARNED, LEARN, LEARN_MARGIN = 0.42, 0.08, 0.25, 0.35, 0.4, 0.12


class Faces:
    def __init__(self):
        import onnxruntime as ort
        z = fetch(FACE_ZIP)
        d = os.path.join(CACHE, 'buffalo_s')
        if not os.path.exists(os.path.join(d, 'w600k_mbf.onnx')):
            with zipfile.ZipFile(z) as f:
                f.extractall(d)
        find = lambda n: next(os.path.join(r, n) for r, _, fs in os.walk(d) if n in fs)
        prov = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if 'CUDAExecutionProvider' in ort.get_available_providers() else ['CPUExecutionProvider']
        self.det = ort.InferenceSession(find('det_500m.onnx'), providers=prov)
        self.rec = ort.InferenceSession(find('w600k_mbf.onnx'), providers=prov)

    def __call__(self, bgr, min_size=None):
        """Every face: dicts with box, kps (5 points), size (eye-to-mouth px), emb (unit 512-d)."""
        h, w = bgr.shape[:2]
        D = 640
        s = D / max(w, h)
        img = np.zeros((D, D, 3), np.uint8)
        img[:round(h * s), :round(w * s)] = cv2.resize(bgr, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)
        x = ((img[:, :, ::-1].astype(np.float32) - 127.5) / 128).transpose(2, 0, 1)[None]
        o = self.det.run(None, {self.det.get_inputs()[0].name: x})
        found = []
        for si, st in enumerate([8, 16, 32]):
            sc, bx, kp = o[si].reshape(-1), o[si + 3].reshape(-1, 4), o[si + 6].reshape(-1, 10)
            gw = D // st
            for a in np.where(sc >= 0.5)[0]:
                cell = a >> 1
                cx, cy = (cell % gw) * st, (cell // gw) * st
                b = np.array([cx - bx[a, 0] * st, cy - bx[a, 1] * st, cx + bx[a, 2] * st, cy + bx[a, 3] * st]) / s
                k = np.array([[cx + kp[a, 2 * p] * st, cy + kp[a, 2 * p + 1] * st] for p in range(5)]) / s
                found.append(dict(box=b, kps=k, score=float(sc[a])))
        found.sort(key=lambda f: -f['score'])
        kept = []
        for f in found:
            if all(_iou(k['box'], f['box']) < 0.4 for k in kept):
                kept.append(f)
        min_size = min_size or max(w, h) * 0.012
        kept = [f for f in kept if min(f['box'][2] - f['box'][0], f['box'][3] - f['box'][1]) >= min_size]
        if not kept:
            return []
        crops = []
        for f in kept:
            M, _ = cv2.estimateAffinePartial2D(f['kps'].astype(np.float32), TEMPLATE, method=cv2.LMEDS)
            c = cv2.warpAffine(bgr, M, (112, 112), borderValue=0)
            crops.append(((c[:, :, ::-1].astype(np.float32) - 127.5) / 127.5).transpose(2, 0, 1))
        e = self.rec.run(None, {self.rec.get_inputs()[0].name: np.stack(crops)})[0]
        e /= np.linalg.norm(e, axis=1, keepdims=True)
        for f, v in zip(kept, e):
            f['emb'] = v
            f['size'] = float((f['kps'][3][1] + f['kps'][4][1]) / 2 - (f['kps'][0][1] + f['kps'][1][1]) / 2)
        return kept


def _iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    i = ix * iy
    return i / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - i + 1e-9)


def frontal(f):
    le, re, nose = f['kps'][0], f['kps'][1], f['kps'][2]
    eyes = np.linalg.norm(re - le)
    t = float(np.dot(nose - le, re - le) / (eyes * eyes + 1e-6))
    return eyes > 0.3 * (f['box'][2] - f['box'][0]) and 0.25 < t < 0.75


class Matcher:
    """Her face (see js/face.js matcher): photos, plus how she looks in this video once sure."""

    def __init__(self, ref):
        self.ref, self.sum, self.seen, self.video = ref, np.zeros(512, np.float32), 0, None

    def learn(self, emb):
        self.sum += emb
        self.seen += 1
        self.video = self.sum / np.linalg.norm(self.sum)

    def find(self, found, plausible=lambda f: True):
        best, second = None, -1
        for f in found:
            f['photo'] = float(f['emb'] @ self.ref)
            f['sim'] = max(f['photo'], float(f['emb'] @ self.video)) if self.seen >= 2 else f['photo']
            if best is None or f['sim'] > best['sim']:
                if best is not None:
                    second = best['sim']
                best = f
            elif f['sim'] > second:
                second = f['sim']
        her = best if best is not None and best['sim'] >= SAME and best['sim'] - second >= MARGIN else None
        if her is not None and not plausible(her):
            her = None
        if her is not None and her['photo'] >= LEARN and her['sim'] - second >= LEARN_MARGIN:
            self.learn(her['emb'])
        return her

    def other(self, f):
        return f['sim'] < (OTHER_LEARNED if self.seen >= 5 else OTHER)


def reference_from_photos(faces, paths):
    embs = []
    for p in paths:
        img = cv2.imdecode(np.fromfile(p, np.uint8), cv2.IMREAD_COLOR)
        if img is None:
            continue
        fs = faces(img, min_size=24)
        if fs:
            embs.append(max(fs, key=lambda f: f['box'][2] - f['box'][0])['emb'])
    if not embs:
        return None
    v = np.mean(embs, 0)
    print(f'사진 {len(embs)}장에서 얼굴을 찾았어요.')
    return v / np.linalg.norm(v)


def reference_from_video(faces, path, samples=200):
    """No photos: in a fancam the member it follows is the face seen most often. Over the whole
    video, not just the part being cut: in a few seconds another member can outnumber her."""
    cap = cv2.VideoCapture(path)
    start, end = 0.0, probe(path)[2]
    embs, sizes = [], []
    for t in np.linspace(start, end, samples, endpoint=False):
        cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
        ok, fr = cap.read()
        if not ok:
            continue
        for f in faces(fr):
            if frontal(f):
                embs.append(f['emb'])
                sizes.append(f['size'])
    if not embs:
        return None
    E = np.stack(embs)
    clusters = []
    for i in np.argsort(sizes)[::-1]:
        best, bs = None, 0.35
        for c in clusters:
            s = float(c['c'] @ E[i])
            if s > bs:
                best, bs = c, s
        if best is None:
            clusters.append(dict(c=E[i].copy(), m=[i]))
        else:
            best['m'].append(i)
            v = E[best['m']].sum(0)
            best['c'] = v / np.linalg.norm(v)
    big = max(clusters, key=lambda c: len(c['m']))
    print(f'사진이 없어서 영상에서 가장 많이 나온 얼굴로 정했어요 ({len(big["m"])}/{len(embs)}번).')
    return big['c']


# ---------------------------------------------------------------------------
# Video in and out

class Frames:
    """Frames at output times start + (i + 0.5) / fps, at `size` (w, h), read once and cached."""

    def __init__(self, path, start, end, fps, size):
        self.cap = cv2.VideoCapture(path)
        self.src_fps = self.cap.get(cv2.CAP_PROP_FPS) or 30
        self.cap.set(cv2.CAP_PROP_POS_MSEC, max(0, start - 1) * 1000)
        self.pos = int(round(self.cap.get(cv2.CAP_PROP_POS_FRAMES)))
        self.start, self.fps, self.size = start, fps, size
        self.total = max(1, int((end - start) * fps))
        self.cache, self.last = {}, None

    def get(self, i):
        if i in self.cache:
            return self.cache[i]
        want = int((self.start + (i + 0.5) / self.fps) * self.src_fps)
        while self.pos <= want:
            ok, fr = self.cap.read()
            if not ok:
                break
            self.pos += 1
            if self.pos > want:
                self.last = fr
        if self.last is None:
            ok, self.last = self.cap.read()
        img = cv2.resize(self.last, self.size, interpolation=cv2.INTER_AREA)
        self.cache[i] = img
        return img

    def forget_before(self, i):
        for k in [k for k in self.cache if k < i]:
            del self.cache[k]


def probe(path):
    cap = cv2.VideoCapture(path)
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    n, fps = cap.get(cv2.CAP_PROP_FRAME_COUNT), cap.get(cv2.CAP_PROP_FPS) or 30
    return w, h, n / fps


def even(v):
    return max(2, int(round(v / 2)) * 2)


# ---------------------------------------------------------------------------
# Main

def make_clip(video, out_dir, name='설윤', photos=(), start=0.0, end=None, fps=30, max_side=1280,
              shape='cutout', sam_model='base_plus', chunk=300, face_every=0.25, crop=True):
    from sam2.build_sam import build_sam2_video_predictor
    import torchvision

    vw, vh, dur = probe(video)
    end = min(end or dur, dur)
    s = max_side / max(vw, vh)
    W, H = even(vw * s), even(vh * s)  # working size
    frames = Frames(video, start, end, fps, (W, H))
    total = frames.total
    print(f'{vw}x{vh} → {W}x{H}, {end - start:.1f}초, {total}프레임, 장치: {DEVICE}')

    faces = Faces()
    ref = reference_from_photos(faces, photos) if photos else None
    if ref is None:
        ref = reference_from_video(faces, video)
    if ref is None:
        raise RuntimeError('영상에서 얼굴을 찾지 못했어요. 설윤 사진을 넣어 주세요.')
    her_face = Matcher(ref)

    people = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(weights='DEFAULT').to(DEVICE).eval()

    def persons(bgr):
        with torch.no_grad():
            x = torch.from_numpy(bgr[:, :, ::-1].copy()).permute(2, 0, 1).float().div(255).to(DEVICE)
            r = people([x])[0]
        keep = (r['labels'] == 1) & (r['scores'] > 0.6)
        return r['boxes'][keep].cpu().numpy()

    def near_middle(f):
        x = (f['box'][0] + f['box'][2]) / 2 / W
        return 0.2 < x < 0.8

    def look(bgr):
        found = faces(bgr)
        return found, her_face.find(found, lambda f: near_middle(f) or f['sim'] >= 0.55)

    def prompt_from_face(bgr, her, found):
        """Box around her body (from the person detector), her face, and her body below it."""
        fx, fy = (her['box'][0] + her['box'][2]) / 2, (her['box'][1] + her['box'][3]) / 2
        fh = her['box'][3] - her['box'][1]
        pts, labels, box = [[fx, fy]], [1], None
        cands = [b for b in persons(bgr) if b[0] <= fx <= b[2] and b[1] - fh <= fy <= b[1] + (b[3] - b[1]) * 0.4]
        if cands:
            box = min(cands, key=lambda b: (b[2] - b[0]) * (b[3] - b[1]))
        for dy in (2.5, 5, 7.5):
            if fy + dy * fh < H * 0.98:
                pts.append([fx, fy + dy * fh]); labels.append(1)
        for f in found:
            if f is not her:
                pts.append([(f['box'][0] + f['box'][2]) / 2, (f['box'][1] + f['box'][3]) / 2]); labels.append(0)
        return np.array(pts, np.float32), np.array(labels, np.int32), (np.array(box, np.float32) if box is not None else None)

    # Find her at the start (or up to 3 s in, where her face shows).
    first = None
    for i in range(0, min(total, int(3 * fps) + 1), max(1, int(fps / 2))):
        found, her = look(frames.get(i))
        if her is not None:
            first = (i, her, found)
            break
    if first is None:
        raise RuntimeError('처음 3초 안에서 설윤 얼굴을 찾지 못했어요. START를 얼굴이 보이는 곳으로 옮겨 주세요.')
    skip, her, found = first
    if skip:
        print(f'처음 {skip / fps:.1f}초는 얼굴이 안 보여서 거기서부터 시작해요.')
    prompt = ('points', prompt_from_face(frames.get(skip), her, found))

    # Matting model
    rvm = torch.hub.load('PeterL1n/RobustVideoMatting', 'resnet50' if DEVICE == 'cuda' else 'mobilenetv3', trust_repo=True).to(DEVICE).eval()
    rec = [None] * 4
    ratio = min(1.0, 512 / max(W, H))

    cfg, url = SAM_MODELS[sam_model]
    predictor = build_sam2_video_predictor(cfg, fetch(url), device=DEVICE)

    # Output: colour and alpha packed like the app does (side by side for portrait, else stacked).
    if shape == 'circle':
        side = even(min(W, H, 1080))
        ow, oh = side, side
    else:
        ow, oh = W, H
    layout = 'sbs' if oh >= ow else 'tb'
    PW, PH = (ow * 2, oh) if layout == 'sbs' else (ow, oh * 2)
    os.makedirs(out_dir, exist_ok=True)
    safe = ''.join('_' if c in '\\/:*?"<>|' else c for c in name)
    tmp_out = os.path.join(out_dir, f'.{safe}.tmp.mp4')
    first_t = start + skip / fps
    ff = subprocess.Popen([
        'ffmpeg', '-y', '-loglevel', 'error',
        '-f', 'rawvideo', '-pix_fmt', 'bgr24', '-s', f'{PW}x{PH}', '-r', str(fps), '-i', '-',
        '-ss', f'{first_t:.3f}', '-t', f'{end - first_t:.3f}', '-i', video,
        '-map', '0:v', '-map', '1:a?', '-c:v', 'libx264', '-crf', '17', '-preset', 'medium', '-pix_fmt', 'yuv420p',
        '-c:a', 'aac', '-b:a', '192k', '-shortest', '-movflags', '+faststart', tmp_out,
    ], stdin=subprocess.PIPE)

    side_px = max(W, H)
    k_area = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * int(side_px * 0.015) + 1,) * 2)
    k_inner = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * int(side_px * 0.01) + 1,) * 2)
    bounds = [1.0, 1.0, 0.0, 0.0]
    cam = None
    disk = None
    if shape == 'circle':
        disk = np.zeros((oh, ow), np.float32)
        cv2.circle(disk, (ow // 2, oh // 2), ow // 2 - 2, 1.0, -1, lineType=cv2.LINE_AA)
        disk = cv2.GaussianBlur(disk, (5, 5), 0)

    def write(i, core, hidden):
        nonlocal rec, cam
        bgr = frames.get(i)
        with torch.no_grad():
            src = torch.from_numpy(bgr[:, :, ::-1].copy()).permute(2, 0, 1)[None].float().div(255).to(DEVICE)
            _, pha, *rec = rvm(src, *rec, downsample_ratio=ratio)
        pha = pha[0, 0].float().cpu().numpy()
        if hidden or core is None:
            a = np.zeros((H, W), np.float32)
        else:
            m = core.astype(np.uint8)
            area = cv2.GaussianBlur(cv2.dilate(m, k_area).astype(np.float32), (0, 0), side_px * 0.004)
            inner = cv2.erode(m, k_inner).astype(np.float32)
            a = np.maximum(pha * area, inner)
            a = np.clip((a - 0.1) / 0.85, 0, 1)
        if shape == 'circle':
            ys, xs = np.nonzero(a[::2, ::2] > 0.5)
            if len(xs) > 20:
                xs, ys = xs * 2, ys * 2
                want = (xs.mean(), ys.min() + (ys.max() - ys.min()) * 0.42,
                        min(max((xs.max() - xs.min()) * 0.7, (ys.max() - ys.min()) * 0.55, min(W, H) * 0.12), min(W, H) / 2))
                cam = list(want) if cam is None else [cam[0] + (want[0] - cam[0]) * 0.2, cam[1] + (want[1] - cam[1]) * 0.2, cam[2] + (want[2] - cam[2]) * 0.08]
            colour = np.zeros((oh, ow, 3), np.uint8)
            alpha = np.zeros((oh, ow), np.float32)
            if cam is not None:
                rad = cam[2]
                cx, cy = min(max(cam[0], rad), W - rad), min(max(cam[1], rad), H - rad)
                M = np.array([[ow / (2 * rad), 0, -(cx - rad) * ow / (2 * rad)], [0, oh / (2 * rad), -(cy - rad) * oh / (2 * rad)]], np.float32)
                colour = cv2.warpAffine(bgr, M, (ow, oh), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
                alpha = disk * (0.0 if hidden else 1.0)
        else:
            colour, alpha = bgr, a
            ys, xs = np.nonzero(a[::2, ::2] > 0.5)
            if len(xs):
                bounds[0] = min(bounds[0], xs.min() * 2 / W); bounds[1] = min(bounds[1], ys.min() * 2 / H)
                bounds[2] = max(bounds[2], xs.max() * 2 / W); bounds[3] = max(bounds[3], ys.max() * 2 / H)
        a8 = (alpha * 255).astype(np.uint8)
        packed = np.zeros((PH, PW, 3), np.uint8)
        if layout == 'sbs':
            packed[:, :ow] = colour; packed[:, ow:] = a8[..., None]
        else:
            packed[:oh] = colour; packed[oh:] = a8[..., None]
        ff.stdin.write(packed.tobytes())

    # SAM 2 frames for one chunk, written as JPEGs (what its video predictor reads)
    jdir = os.path.join(CACHE, 'chunk')
    sam_scale = min(1.0, 1024 / max(W, H))
    SW, SH = even(W * sam_scale), even(H * sam_scale)

    every = max(1, int(round(fps * face_every)))
    fixes, i0 = 0, skip
    off_face, last_her, doubted = 0, None, False
    from tqdm.auto import tqdm
    bar = tqdm(total=total - skip, unit='프레임')
    while i0 < total:
        i1 = min(total, i0 + chunk)
        shutil.rmtree(jdir, ignore_errors=True)
        os.makedirs(jdir)
        for k in range(i0, i1):
            cv2.imwrite(os.path.join(jdir, f'{k - i0:05d}.jpg'), cv2.resize(frames.get(k), (SW, SH), interpolation=cv2.INTER_AREA), [cv2.IMWRITE_JPEG_QUALITY, 92])
        with torch.inference_mode(), torch.autocast(DEVICE, dtype=torch.bfloat16, enabled=BF16):
            state = predictor.init_state(video_path=jdir, offload_video_to_cpu=True, offload_state_to_cpu=DEVICE == 'cuda')
            kind, p = prompt
            if kind == 'mask':
                predictor.add_new_mask(state, frame_idx=0, obj_id=1, mask=cv2.resize(p.astype(np.uint8), (SW, SH), interpolation=cv2.INTER_NEAREST) > 0)
            else:
                pts, labels, box = p
                predictor.add_new_points_or_box(state, frame_idx=0, obj_id=1, points=pts * sam_scale, labels=labels,
                                                box=box * sam_scale if box is not None else None)
            masks = {}
            for fidx, _, logits in predictor.propagate_in_video(state):
                masks[fidx] = cv2.resize(logits[0, 0].float().cpu().numpy(), (W, H), interpolation=cv2.INTER_LINEAR) > 0
            predictor.reset_state(state)

        # Face checks over the chunk: where the tracker left her, end the chunk and start again.
        stop, next_prompt = i1 - i0, None
        hidden = np.zeros(i1 - i0, bool)
        hide = False
        for k in range(i1 - i0):
            i = i0 + k
            if i % every == 0:
                found, her = look(frames.get(i))
                core = masks.get(k)
                in_mask = lambda f: core is not None and np.mean([core[int(min(H - 1, max(0, y))), int(min(W - 1, max(0, x)))] for x, y in f['kps'][:3]]) >= 0.6
                strangers = [f for f in found if f is not her and her_face.other(f) and frontal(f) and f['size'] > max(W, H) * 0.024 and in_mask(f)]
                if her is not None:
                    hide, doubted = False, False
                    last_her = ((her['box'][0] + her['box'][2]) / 2, (her['box'][1] + her['box'][3]) / 2, her['box'][2] - her['box'][0], i)
                    off_face = off_face + 1 if not in_mask(her) else 0
                    apart = [f for f in strangers if math.hypot((f['box'][0] + f['box'][2]) / 2 - last_her[0], (f['box'][1] + f['box'][3]) / 2 - last_her[1]) > 1.5 * last_her[2]]
                    if (off_face >= 2 or apart) and k > 0:
                        off_face = 0
                        stop, next_prompt = k, ('points', prompt_from_face(frames.get(i), her, found))
                        fixes += 1
                        break
                elif strangers:
                    near = lambda f: last_her is not None and i - last_her[3] <= fps and math.hypot((f['box'][0] + f['box'][2]) / 2 - last_her[0], (f['box'][1] + f['box'][3]) / 2 - last_her[1]) < 1.5 * last_her[2]
                    if all(near(f) for f in strangers) and not doubted:
                        doubted = True
                    else:
                        hide = True
                else:
                    doubted = False
            hidden[k] = hide
        for k in range(stop):
            write(i0 + k, masks.get(k), hidden[k])
            bar.update(1)
        last_mask = masks.get(stop - 1) if stop > 0 else None
        prompt = next_prompt or (('mask', last_mask) if last_mask is not None and last_mask.any() else prompt)
        frames.forget_before(i0 + stop)
        i0 += stop if stop > 0 else 1
    bar.close()
    ff.stdin.close()
    ff.wait()
    shutil.rmtree(jdir, ignore_errors=True)
    print(f'얼굴로 {fixes}번 다시 잡았어요.')

    # Crop the empty space around her (both halves alike), like the app's player does.
    final = os.path.join(out_dir, f'{safe}.stage-{layout}.mp4')
    if crop and shape != 'circle' and bounds[2] > bounds[0]:
        pad = 0.03
        x0, y0 = max(0, bounds[0] - pad), max(0, bounds[1] - pad)
        x1, y1 = min(1, bounds[2] + pad), min(1, bounds[3] + pad)
        cx, cy, cw, ch = even(x0 * ow) // 2 * 2, even(y0 * oh) // 2 * 2, even((x1 - x0) * ow), even((y1 - y0) * oh)
        cw, ch = min(cw, ow - cx), min(ch, oh - cy)
        if layout == 'sbs':
            fc = f'[0:v]split[a][b];[a]crop={cw}:{ch}:{cx}:{cy}[c];[b]crop={cw}:{ch}:{ow + cx}:{cy}[d];[c][d]hstack[v]'
        else:
            fc = f'[0:v]split[a][b];[a]crop={cw}:{ch}:{cx}:{cy}[c];[b]crop={cw}:{ch}:{cx}:{oh + cy}[d];[c][d]vstack[v]'
        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', tmp_out, '-filter_complex', fc, '-map', '[v]', '-map', '0:a?',
                        '-c:v', 'libx264', '-crf', '17', '-preset', 'medium', '-pix_fmt', 'yuv420p', '-c:a', 'copy',
                        '-movflags', '+faststart', final], check=True)
        os.remove(tmp_out)
    else:
        os.replace(tmp_out, final)
    print('완성:', final)
    return final


def download_youtube(url, out_dir):
    """Best quality up to 4K (the app's analysis stops at its own size anyway)."""
    os.makedirs(out_dir, exist_ok=True)
    out = os.path.join(out_dir, '%(title).80s.%(ext)s')
    r = subprocess.run(['yt-dlp', '-f', 'bv*[height<=2160]+ba/b', '--merge-output-format', 'mp4', '-o', out,
                        '--print', 'after_move:filepath', url], check=True, capture_output=True, text=True)
    return r.stdout.strip().splitlines()[-1]


if __name__ == '__main__':
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument('video'); ap.add_argument('out_dir')
    ap.add_argument('--name', default='설윤'); ap.add_argument('--photos', nargs='*', default=[])
    ap.add_argument('--start', type=float, default=0); ap.add_argument('--end', type=float)
    ap.add_argument('--fps', type=int, default=30); ap.add_argument('--max-side', type=int, default=1280)
    ap.add_argument('--shape', default='cutout'); ap.add_argument('--sam', default='base_plus')
    ap.add_argument('--chunk', type=int, default=300)
    a = ap.parse_args()
    make_clip(a.video, a.out_dir, a.name, a.photos, a.start, a.end, a.fps, a.max_side, a.shape, a.sam, a.chunk)

## 5. 실행

처리하는 동안 이 탭을 열어 두세요. 진행 막대가 보여요.

In [ ]:
import glob, importlib, sys
sys.path.insert(0, '/content')
import fancam
importlib.reload(fancam)

video = VIDEO.strip()
if video.startswith('http'):
    video = fancam.download_youtube(video, '/content/videos')
    print('받은 영상:', video)
photos = sorted(p for p in glob.glob(os.path.join(PHOTOS_FOLDER, '*')) if p.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))) if PHOTOS_FOLDER else []
max_side = {'보통 (긴 변 960)': 960, '고화질 (긴 변 1280)': 1280, 'FHD (긴 변 1920)': 1920}[QUALITY]
out = fancam.make_clip(video, OUT_FOLDER, name=NAME, photos=photos, start=float(START), end=float(END) or None,
                       fps=int(FPS), max_side=max_side, shape='circle' if SHAPE == '원' else 'cutout', sam_model=SAM_MODEL)

## 6. 앱에 넣기

드라이브의 **직캠 스테이지/코랩** 폴더에서 `이름.stage-….mp4` 파일을 받아, 앱 보관함의 **파일 가져오기**로 열면 돼요.
이름 끝의 `.stage-tb`/`.stage-sbs`는 앱이 파일을 읽는 데 필요하니 바꾸지 마세요.